# 03 — MinHash and Locality-Sensitive Hashing

In this notebook, we identify games with similar player audiences.

Each game is represented by the set of users who have played it for at least 2 hours. We first explore exact Jaccard similarity and then implement MinHash and Locality-Sensitive Hashing (LSH) to find similar games efficiently.

In [2]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/processed/interactions_filtered.parquet")

interactions = pd.read_parquet(DATA_PATH)

print("Interactions:", len(interactions))
print("Users:", interactions["user_id"].nunique())
print("Games:", interactions["game_id"].nunique())

interactions.head()

Interactions: 1871351
Users: 66754
Games: 2402


,user_id,game_id,game_name,playtime_forever,playtime_2weeks
0,wolop,4000,Garry's Mod,642773,7299
1,Evilutional,212200,Mabinogi,635295,0
2,76561198019826668,42710,Call of Duty: Black Ops - Multiplayer,632452,8126
3,76561198039832932,4000,Garry's Mod,613411,588
4,tsunamitad,72200,Universe Sandbox,600068,0


## Represent each game as a set of users

To compare games based on player behaviour, we represent each game by the set of users who are engaged with it.

For example:

- Game A → {User 1, User 2, User 3}
- Game B → {User 2, User 3, User 4}

These sets can then be compared using Jaccard similarity.

In [3]:
game_users = (
    interactions
    .groupby("game_id")["user_id"]
    .apply(set)
    .to_dict()
)

print("Number of games:", len(game_users))

# Show the number of engaged users for a few games.
for game_id, users in list(game_users.items())[:5]:
    print(game_id, "->", len(users), "users")

Number of games: 2402
10 -> 2511 users
100 -> 517 users
10090 -> 3036 users
10130 -> 78 users
10150 -> 771 users


## Exact Jaccard similarity

Jaccard similarity measures how similar two sets are.

For two games, we compare their sets of engaged users:

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

A Jaccard similarity of:

- `0` means the games share no engaged users
- `1` means they have exactly the same engaged users
- values in between indicate partial overlap

In [4]:
def jaccard_similarity(users_a, users_b):
    """Calculate the Jaccard similarity between two sets of users."""
    intersection = users_a & users_b
    union = users_a | users_b

    return len(intersection) / len(union)


game_ids = list(game_users.keys())

game_a = game_ids[0]
game_b = game_ids[1]

similarity = jaccard_similarity(
    game_users[game_a],
    game_users[game_b]
)

print("Game A:", game_a)
print("Game B:", game_b)
print("Jaccard similarity:", similarity)

Game A: 10
Game B: 100
Jaccard similarity: 0.14393653192293163


In [5]:
users_a = game_users[game_a]
users_b = game_users[game_b]

intersection = users_a & users_b
union = users_a | users_b

print("Users in Game A:", len(users_a))
print("Users in Game B:", len(users_b))
print("Shared users:", len(intersection))
print("Users in union:", len(union))
print("Jaccard similarity:", len(intersection) / len(union))

Users in Game A: 2511
Users in Game B: 517
Shared users: 381
Users in union: 2647
Jaccard similarity: 0.14393653192293163


In [6]:
def compare_games(game_a, game_b):
    """Print overlap statistics for two games."""
    users_a = game_users[game_a]
    users_b = game_users[game_b]

    intersection = users_a & users_b
    union = users_a | users_b

    similarity = len(intersection) / len(union)

    print("Game A:", game_a)
    print("Game B:", game_b)
    print("Users in Game A:", len(users_a))
    print("Users in Game B:", len(users_b))
    print("Shared users:", len(intersection))
    print("Jaccard similarity:", similarity)

In [7]:
compare_games("10", "80")

Game A: 10
Game B: 80
Users in Game A: 2511
Users in Game B: 1149
Shared users: 750
Jaccard similarity: 0.25773195876288657


## Why not compare every game pair directly?

Exact Jaccard similarity works well for comparing individual games, but comparing every possible pair becomes expensive.

With 2,402 games, there are almost 2.9 million unique game pairs.

MinHash gives us a compact signature for each game's user set that approximately preserves Jaccard similarity. LSH can then use these signatures to quickly identify promising candidate pairs.

The overall idea is:

1. Convert each game into a user set
2. Create a MinHash signature for each game
3. Use LSH to find likely similar game pairs
4. Compute exact Jaccard similarity only for those candidates

In [8]:
number_of_games = len(game_users)

number_of_pairs = (
    number_of_games * (number_of_games - 1)
) // 2

print("Number of games:", number_of_games)
print("Possible game pairs:", number_of_pairs)

Number of games: 2402
Possible game pairs: 2883601


In [9]:
import random
import time


def jaccard_similarity_fast(users_a, users_b):
    """Calculate Jaccard similarity without explicitly building the union."""
    intersection_size = len(users_a & users_b)
    union_size = len(users_a) + len(users_b) - intersection_size

    return intersection_size / union_size


game_ids = list(game_users.keys())

# Generate 100,000 random game pairs for the benchmark.
number_of_test_pairs = 100_000

test_pairs = [
    random.sample(game_ids, 2)
    for _ in range(number_of_test_pairs)
]

start_time = time.perf_counter()

for game_a, game_b in test_pairs:
    jaccard_similarity_fast(
        game_users[game_a],
        game_users[game_b]
    )

elapsed_time = time.perf_counter() - start_time

comparisons_per_second = number_of_test_pairs / elapsed_time
estimated_full_time = number_of_pairs / comparisons_per_second

print(f"Test comparisons: {number_of_test_pairs:,}")
print(f"Benchmark time: {elapsed_time:.2f} seconds")
print(f"Comparisons per second: {comparisons_per_second:,.0f}")
print(f"Estimated time for all {number_of_pairs:,} pairs: {estimated_full_time:.2f} seconds")

Test comparisons: 100,000
Benchmark time: 0.98 seconds
Comparisons per second: 101,526
Estimated time for all 2,883,601 pairs: 28.40 seconds


### Benchmark interpretation

On the current filtered dataset, 100,000 exact Jaccard comparisons took approximately 0.69 seconds, corresponding to roughly 144,000 comparisons per second. This suggests that exhaustively comparing all 2.88 million game pairs would take around 20 seconds on the current machine.

The earlier filtering step also has a substantial computational impact. The original interaction dataset contains 10,978 unique games, corresponding to more than 60 million possible game pairs. After requiring at least 2 hours of playtime and at least 50 engaged players per game, only 2,402 games remain, reducing the number of possible comparisons to approximately 2.88 million. This is roughly a 21-fold reduction.

Therefore, exhaustive Jaccard comparison is feasible for the filtered dataset. However, this is partly a consequence of the filtering, and pairwise comparison still scales quadratically with the number of games.

MinHash and LSH therefore remain useful as a more scalable approach. Since exhaustive Jaccard is feasible at our current scale, we can also use it as a ground-truth baseline to evaluate how effectively MinHash and LSH identify similar game pairs.

## Exhaustive Jaccard baseline

Since exhaustive comparison is feasible for the filtered dataset, we calculate the exact Jaccard similarity for every pair of games.

This provides a ground-truth baseline that can later be used to evaluate the MinHash and LSH results.

In [10]:
import numpy as np
import time


game_ids = list(game_users.keys())
number_of_games = len(game_ids)
number_of_pairs = number_of_games * (number_of_games - 1) // 2

# Store all exact similarities efficiently in a NumPy array.
exact_jaccard_scores = np.empty(number_of_pairs, dtype=np.float32)

start_time = time.perf_counter()

index = 0

for i in range(number_of_games):
    users_a = game_users[game_ids[i]]

    for j in range(i + 1, number_of_games):
        users_b = game_users[game_ids[j]]

        intersection_size = len(users_a & users_b)
        union_size = len(users_a) + len(users_b) - intersection_size

        exact_jaccard_scores[index] = intersection_size / union_size
        index += 1

elapsed_time = time.perf_counter() - start_time

print(f"Comparisons completed: {index:,}")
print(f"Time taken: {elapsed_time:.2f} seconds")

Comparisons completed: 2,883,601
Time taken: 22.10 seconds


## Distribution of exact Jaccard similarities

Before defining which game pairs should be considered similar, we inspect the distribution of exact Jaccard scores across all game pairs.

In [11]:
print("Minimum:", exact_jaccard_scores.min())
print("Median:", np.median(exact_jaccard_scores))
print("Mean:", exact_jaccard_scores.mean())
print("90th percentile:", np.percentile(exact_jaccard_scores, 90))
print("95th percentile:", np.percentile(exact_jaccard_scores, 95))
print("99th percentile:", np.percentile(exact_jaccard_scores, 99))
print("Maximum:", exact_jaccard_scores.max())

Minimum: 0.0
Median: 0.0057471264
Mean: 0.015009959
90th percentile: 0.03550296
95th percentile: 0.0621442
99th percentile: 0.16
Maximum: 0.8939394


In [12]:
thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]

for threshold in thresholds:
    count = (exact_jaccard_scores >= threshold).sum()
    percentage = count / len(exact_jaccard_scores) * 100

    print(
        f"Jaccard >= {threshold:.2f}: "
        f"{count:,} pairs ({percentage:.2f}%)"
    )

Jaccard >= 0.05: 192,617 pairs (6.68%)
Jaccard >= 0.10: 71,297 pairs (2.47%)
Jaccard >= 0.15: 33,372 pairs (1.16%)
Jaccard >= 0.20: 16,066 pairs (0.56%)
Jaccard >= 0.30: 3,742 pairs (0.13%)
Jaccard >= 0.50: 206 pairs (0.01%)


## MinHash intuition

Exact Jaccard similarity requires comparing the full user sets of two games.

MinHash instead creates a compact signature for each set. The key property is that two sets with high Jaccard similarity are likely to have similar MinHash signatures.

We start with a single hash function before extending this to a full signature.

## One MinHash function

To apply MinHash, we first map each user ID to an integer.

We then apply a hash function to every user in a game's user set and keep the smallest resulting hash value. This minimum value is the game's MinHash value for that hash function.

In [13]:
# Give every unique user a reproducible integer ID.
unique_users = sorted(interactions["user_id"].unique())

user_to_int = {
    user_id: index
    for index, user_id in enumerate(unique_users)
}

print("Number of users:", len(user_to_int))

Number of users: 66754


In [14]:
# Parameters for one simple hash function.
a = 37
b = 17
prime = 100_003


def hash_user(user_int):
    """Hash an integer user ID."""
    return (a * user_int + b) % prime


def minhash_value(users):
    """Return the minimum hash value for a set of users."""
    hashed_users = [
        hash_user(user_to_int[user])
        for user in users
    ]

    return min(hashed_users)


game_id = game_ids[0]

value = minhash_value(game_users[game_id])

print("Game:", game_id)
print("Number of users:", len(game_users[game_id]))
print("MinHash value:", value)

Game: 10
Number of users: 2511
MinHash value: 99


In [15]:
game_a = game_ids[0]
game_b = game_ids[1]

minhash_a = minhash_value(game_users[game_a])
minhash_b = minhash_value(game_users[game_b])

print("Game A:", game_a, "MinHash:", minhash_a)
print("Game B:", game_b, "MinHash:", minhash_b)
print("Same MinHash:", minhash_a == minhash_b)

Game A: 10 MinHash: 99
Game B: 100 MinHash: 546
Same MinHash: False


## MinHash signatures

A single MinHash value is too noisy to estimate similarity reliably.

Instead, we use many different hash functions. Each hash function produces one minimum value, and together these values form a MinHash signature.

The similarity between two signatures is estimated as the fraction of positions where the two signatures are equal.

In [16]:
import numpy as np


NUM_HASHES = 100
PRIME = 100_003

# Create reproducible random hash functions.
rng = np.random.default_rng(seed=42)

a_values = rng.integers(
    1,
    PRIME,
    size=NUM_HASHES
)

b_values = rng.integers(
    0,
    PRIME,
    size=NUM_HASHES
)


def create_minhash_signature(users):
    """Create a MinHash signature for a set of users."""
    user_integers = np.array([
        user_to_int[user]
        for user in users
    ])

    signature = []

    for a, b in zip(a_values, b_values):
        hashed_users = (a * user_integers + b) % PRIME
        signature.append(hashed_users.min())

    return np.array(signature)

In [17]:
signature_a = create_minhash_signature(game_users[game_a])
signature_b = create_minhash_signature(game_users[game_b])

estimated_similarity = np.mean(signature_a == signature_b)

exact_similarity = jaccard_similarity(
    game_users[game_a],
    game_users[game_b]
)

print("Exact Jaccard similarity:", exact_similarity)
print("MinHash estimate:", estimated_similarity)
print("Matching signature positions:", (signature_a == signature_b).sum())
print("Number of hash functions:", NUM_HASHES)

Exact Jaccard similarity: 0.14393653192293163
MinHash estimate: 0.08
Matching signature positions: 8
Number of hash functions: 100


In [18]:
for num_hashes in [50, 100, 200, 500]:
    rng = np.random.default_rng(seed=42)

    a_values_test = rng.integers(
        1,
        PRIME,
        size=num_hashes
    )

    b_values_test = rng.integers(
        0,
        PRIME,
        size=num_hashes
    )

    def create_signature_with_hashes(users):
        user_integers = np.array([
            user_to_int[user]
            for user in users
        ])

        signature = []

        for a, b in zip(a_values_test, b_values_test):
            hashed_users = (a * user_integers + b) % PRIME
            signature.append(hashed_users.min())

        return np.array(signature)

    signature_a = create_signature_with_hashes(game_users[game_a])
    signature_b = create_signature_with_hashes(game_users[game_b])

    estimate = np.mean(signature_a == signature_b)

    print(
        f"{num_hashes} hashes -> "
        f"estimated similarity: {estimate:.3f}"
    )

50 hashes -> estimated similarity: 0.140
100 hashes -> estimated similarity: 0.080
200 hashes -> estimated similarity: 0.165
500 hashes -> estimated similarity: 0.148


## Generate MinHash signatures for all games

We now create a MinHash signature for every game in the filtered dataset.

We use 200 hash functions as a balance between approximation accuracy and computational cost. These signatures will later be used by LSH to identify candidate pairs without comparing every pair directly.

In [19]:
NUM_HASHES = 200
PRIME = 100_003

rng = np.random.default_rng(seed=42)

a_values = rng.integers(
    1,
    PRIME,
    size=NUM_HASHES
)

b_values = rng.integers(
    0,
    PRIME,
    size=NUM_HASHES
)


def create_minhash_signature(users):
    """Create a MinHash signature for a set of users."""
    user_integers = np.array([
        user_to_int[user]
        for user in users
    ])

    signature = np.empty(NUM_HASHES, dtype=np.int64)

    for i, (a, b) in enumerate(zip(a_values, b_values)):
        hashed_users = (a * user_integers + b) % PRIME
        signature[i] = hashed_users.min()

    return signature

In [20]:
start_time = time.perf_counter()

minhash_signatures = {}

for game_id, users in game_users.items():
    minhash_signatures[game_id] = create_minhash_signature(users)

elapsed_time = time.perf_counter() - start_time

print("Games processed:", len(minhash_signatures))
print("Signature length:", NUM_HASHES)
print(f"Time taken: {elapsed_time:.2f} seconds")

Games processed: 2402
Signature length: 200
Time taken: 5.55 seconds


## Locality-Sensitive Hashing

MinHash signatures allow us to approximate Jaccard similarity, but comparing every pair of signatures would still require all pairwise comparisons.

Locality-Sensitive Hashing (LSH) reduces the number of comparisons by dividing each MinHash signature into bands.

Games that have an identical signature within at least one band are considered candidate pairs. Exact Jaccard similarity can then be calculated only for these candidates.

In [21]:
NUM_BANDS = 100
ROWS_PER_BAND = NUM_HASHES // NUM_BANDS

game_id = game_ids[0]
signature = minhash_signatures[game_id]

bands = [
    signature[
        band * ROWS_PER_BAND:
        (band + 1) * ROWS_PER_BAND
    ]
    for band in range(NUM_BANDS)
]

print("Game:", game_id)
print("Signature length:", len(signature))
print("Number of bands:", len(bands))
print("Rows per band:", ROWS_PER_BAND)

print("\nFirst 5 bands:")
for band in bands[:5]:
    print(band)

Game: 10
Signature length: 200
Number of bands: 100
Rows per band: 2

First 5 bands:
[ 5 14]
[18 77]
[ 8 21]
[16 13]
[ 34 252]


## Create LSH buckets

For each band, games with identical band values are placed in the same bucket.

If two or more games appear in the same bucket for at least one band, they become candidate pairs for further comparison.

In [22]:
from collections import defaultdict


lsh_buckets = defaultdict(list)

for game_id, signature in minhash_signatures.items():
    for band in range(NUM_BANDS):
        start = band * ROWS_PER_BAND
        end = start + ROWS_PER_BAND

        band_values = tuple(signature[start:end])

        # Include the band number so identical values in different
        # bands are treated as different buckets.
        bucket_key = (band, band_values)

        lsh_buckets[bucket_key].append(game_id)


print("Total buckets:", len(lsh_buckets))

non_singleton_buckets = sum(
    len(games) > 1
    for games in lsh_buckets.values()
)

print("Buckets containing multiple games:", non_singleton_buckets)

Total buckets: 192185
Buckets containing multiple games: 19599


## Generate candidate pairs

Games that appear together in the same LSH bucket are considered candidate pairs.

A pair may collide in more than one band, so we store the pairs in a set to ensure that each game pair only appears once.

In [23]:
from itertools import combinations


candidate_pairs = set()

for games in lsh_buckets.values():
    if len(games) > 1:
        for game_a, game_b in combinations(games, 2):
            # Sort the IDs so (A, B) and (B, A) are treated as the same pair.
            pair = tuple(sorted((game_a, game_b)))
            candidate_pairs.add(pair)


print("Unique candidate pairs:", len(candidate_pairs))
print(
    "Percentage of all possible pairs:",
    f"{len(candidate_pairs) / number_of_pairs * 100:.2f}%"
)

Unique candidate pairs: 134606
Percentage of all possible pairs: 4.67%


## Evaluate LSH against exact Jaccard similarity

LSH reduces the number of candidate pairs substantially, but this is only useful if it retains most genuinely similar game pairs.

Because exhaustive Jaccard comparison is feasible for our dataset, we can use exact similarities as ground truth and measure how many similar pairs were recovered by LSH.

In [24]:
evaluation_thresholds = [0.10, 0.15, 0.20]

ground_truth_pairs = {
    threshold: set()
    for threshold in evaluation_thresholds
}

start_time = time.perf_counter()

for i in range(number_of_games):
    game_a = game_ids[i]
    users_a = game_users[game_a]

    for j in range(i + 1, number_of_games):
        game_b = game_ids[j]
        users_b = game_users[game_b]

        similarity = jaccard_similarity_fast(
            users_a,
            users_b
        )

        pair = tuple(sorted((game_a, game_b)))

        for threshold in evaluation_thresholds:
            if similarity >= threshold:
                ground_truth_pairs[threshold].add(pair)

elapsed_time = time.perf_counter() - start_time

for threshold in evaluation_thresholds:
    print(
        f"Jaccard >= {threshold:.2f}: "
        f"{len(ground_truth_pairs[threshold]):,} pairs"
    )

print(f"\nTime taken: {elapsed_time:.2f} seconds")

Jaccard >= 0.10: 71,297 pairs
Jaccard >= 0.15: 33,372 pairs
Jaccard >= 0.20: 16,066 pairs

Time taken: 23.90 seconds


## LSH recall

We measure recall as the fraction of genuinely similar game pairs that were retained by LSH.

A high recall means that LSH removes many unnecessary comparisons while still keeping most of the pairs we actually care about.

In [25]:
for threshold in evaluation_thresholds:
    true_pairs = ground_truth_pairs[threshold]

    recovered_pairs = true_pairs & candidate_pairs

    recall = len(recovered_pairs) / len(true_pairs)

    print(
        f"Jaccard >= {threshold:.2f}: "
        f"{len(recovered_pairs):,} / {len(true_pairs):,} recovered "
        f"({recall * 100:.2f}% recall)"
    )

Jaccard >= 0.10: 58,058 / 71,297 recovered (81.43% recall)
Jaccard >= 0.15: 31,829 / 33,372 recovered (95.38% recall)
Jaccard >= 0.20: 15,957 / 16,066 recovered (99.32% recall)


### LSH evaluation

Using 100 bands with 2 rows per band, LSH reduced the search space from 2,883,601 possible game pairs to 134,606 candidate pairs, corresponding to only 4.67% of all possible comparisons.

Despite this reduction, LSH retained:

- 81.43% of pairs with Jaccard similarity ≥ 0.10
- 95.38% of pairs with Jaccard similarity ≥ 0.15
- 99.32% of pairs with Jaccard similarity ≥ 0.20

This shows the intended behaviour of LSH: highly similar games are very likely to become candidates, while most unrelated game pairs are discarded.

## Comparison with a standard MinHash/LSH library

To validate our own implementation, we compare it with the `datasketch` library.

Our MinHash and LSH implementation was written from scratch to demonstrate the underlying course concepts. The library implementation provides an additional reference point for checking whether our implementation shows comparable behaviour.

In [26]:
from datasketch import MinHash, MinHashLSH

print("datasketch imported successfully")

datasketch imported successfully


### Compare MinHash estimates

We first compare the exact Jaccard similarity with the estimates produced by our MinHash implementation and the `datasketch` implementation for the same pair of games.

The estimates do not need to be identical because the two implementations use different hash functions, but they should both approximate the same exact Jaccard similarity.

In [27]:
def create_datasketch_minhash(users):
    """Create a datasketch MinHash signature for a set of users."""
    minhash = MinHash(num_perm=NUM_HASHES)

    for user in users:
        minhash.update(user.encode("utf-8"))

    return minhash


game_a = game_ids[0]
game_b = game_ids[1]

# Exact Jaccard similarity.
exact_similarity = jaccard_similarity(
    game_users[game_a],
    game_users[game_b]
)

# Our MinHash estimate.
our_signature_a = minhash_signatures[game_a]
our_signature_b = minhash_signatures[game_b]

our_estimate = np.mean(
    our_signature_a == our_signature_b
)

# datasketch MinHash estimate.
library_minhash_a = create_datasketch_minhash(
    game_users[game_a]
)

library_minhash_b = create_datasketch_minhash(
    game_users[game_b]
)

library_estimate = library_minhash_a.jaccard(
    library_minhash_b
)

print("Game A:", game_a)
print("Game B:", game_b)
print(f"Exact Jaccard:       {exact_similarity:.3f}")
print(f"Our MinHash:         {our_estimate:.3f}")
print(f"datasketch MinHash:  {library_estimate:.3f}")

Game A: 10
Game B: 100
Exact Jaccard:       0.144
Our MinHash:         0.165
datasketch MinHash:  0.200


### Compare MinHash accuracy across multiple game pairs

A single game pair is not enough to evaluate MinHash accuracy because the estimates are probabilistic.

We therefore compare our implementation and `datasketch` across a random sample of game pairs and measure the average absolute error relative to exact Jaccard similarity.

In [28]:
import random


random.seed(42)

sample_pairs = [
    tuple(random.sample(game_ids, 2))
    for _ in range(100)
]

our_errors = []
library_errors = []

for game_a, game_b in sample_pairs:
    users_a = game_users[game_a]
    users_b = game_users[game_b]

    exact = jaccard_similarity_fast(
        users_a,
        users_b
    )

    # Our MinHash estimate.
    our_estimate = np.mean(
        minhash_signatures[game_a]
        == minhash_signatures[game_b]
    )

    # datasketch MinHash estimate.
    library_a = create_datasketch_minhash(users_a)
    library_b = create_datasketch_minhash(users_b)

    library_estimate = library_a.jaccard(library_b)

    our_errors.append(abs(our_estimate - exact))
    library_errors.append(abs(library_estimate - exact))


print(
    "Our mean absolute error:",
    f"{np.mean(our_errors):.4f}"
)

print(
    "datasketch mean absolute error:",
    f"{np.mean(library_errors):.4f}"
)

Our mean absolute error: 0.0064
datasketch mean absolute error: 0.0051


### Compare LSH implementations

We next compare our LSH implementation with `datasketch` using the same configuration:

- 200 MinHash values
- 100 bands
- 2 rows per band

This allows us to compare the number of candidate pairs and their recall against the same exact Jaccard ground truth.

In [29]:
start_time = time.perf_counter()

library_minhashes = {}

library_lsh = MinHashLSH(
    num_perm=NUM_HASHES,
    params=(NUM_BANDS, ROWS_PER_BAND)
)

for game_id, users in game_users.items():
    minhash = create_datasketch_minhash(users)

    library_minhashes[game_id] = minhash
    library_lsh.insert(game_id, minhash)

elapsed_time = time.perf_counter() - start_time

print("Games indexed:", len(library_minhashes))
print(f"Time taken: {elapsed_time:.2f} seconds")

Games indexed: 2402
Time taken: 21.04 seconds


In [30]:
library_candidate_pairs = set()

start_time = time.perf_counter()

for game_id, minhash in library_minhashes.items():
    matches = library_lsh.query(minhash)

    for matched_game in matches:
        if matched_game == game_id:
            continue

        pair = tuple(sorted((game_id, matched_game)))
        library_candidate_pairs.add(pair)

elapsed_time = time.perf_counter() - start_time

print("datasketch candidate pairs:", len(library_candidate_pairs))
print(
    "Percentage of all possible pairs:",
    f"{len(library_candidate_pairs) / number_of_pairs * 100:.2f}%"
)
print(f"Query time: {elapsed_time:.2f} seconds")

datasketch candidate pairs: 135035
Percentage of all possible pairs: 4.68%
Query time: 0.48 seconds


In [31]:
print("Our candidate pairs:", f"{len(candidate_pairs):,}")
print("datasketch candidate pairs:", f"{len(library_candidate_pairs):,}")

print(
    "Our search space retained:",
    f"{len(candidate_pairs) / number_of_pairs * 100:.2f}%"
)

print(
    "datasketch search space retained:",
    f"{len(library_candidate_pairs) / number_of_pairs * 100:.2f}%"
)

Our candidate pairs: 134,606
datasketch candidate pairs: 135,035
Our search space retained: 4.67%
datasketch search space retained: 4.68%


### Compare LSH recall

We compare the recall of our LSH implementation with the `datasketch` implementation using the same exact Jaccard ground truth.

This shows whether both implementations preserve a similar proportion of genuinely similar game pairs.

In [32]:
for threshold in evaluation_thresholds:
    true_pairs = ground_truth_pairs[threshold]

    our_recovered = true_pairs & candidate_pairs
    library_recovered = true_pairs & library_candidate_pairs

    our_recall = len(our_recovered) / len(true_pairs)
    library_recall = len(library_recovered) / len(true_pairs)

    print(f"Jaccard >= {threshold:.2f}")
    print(f"  Our LSH recall:        {our_recall * 100:.2f}%")
    print(f"  datasketch LSH recall: {library_recall * 100:.2f}%")
    print()

Jaccard >= 0.10
  Our LSH recall:        81.43%
  datasketch LSH recall: 83.42%

Jaccard >= 0.15
  Our LSH recall:        95.38%
  datasketch LSH recall: 96.22%

Jaccard >= 0.20
  Our LSH recall:        99.32%
  datasketch LSH recall: 99.46%



### Validation against `datasketch`

Our implementation produces results that are very similar to the established `datasketch` library.

Using the same configuration of 200 MinHash values, 100 bands and 2 rows per band:

- Our implementation generated 134,606 candidate pairs, corresponding to 4.67% of all possible pairs.
- `datasketch` generated 135,035 candidate pairs, corresponding to 4.68% of all possible pairs.

The recall against exact Jaccard similarity was also very similar:

- At Jaccard ≥ 0.10: 81.43% recall for our implementation vs. 83.42% for `datasketch`
- At Jaccard ≥ 0.15: 95.38% vs. 96.22%
- At Jaccard ≥ 0.20: 99.32% vs. 99.46%

The small differences are expected because the implementations use different hash functions. Overall, the comparison indicates that our from-scratch MinHash and LSH implementation behaves consistently with a standard library implementation.

## Create the final similarity table

LSH has reduced the search space to a much smaller set of candidate game pairs.

We now compute the exact Jaccard similarity only for these candidates. The resulting table will contain one row per candidate pair and can later be used to construct the weighted game-similarity graph.

In [33]:
candidate_rows = []

start_time = time.perf_counter()

for game_a, game_b in candidate_pairs:
    similarity = jaccard_similarity_fast(
        game_users[game_a],
        game_users[game_b]
    )

    candidate_rows.append({
        "game_a": game_a,
        "game_b": game_b,
        "jaccard_similarity": similarity
    })

candidate_similarities = pd.DataFrame(candidate_rows)

elapsed_time = time.perf_counter() - start_time

print("Candidate pairs scored:", len(candidate_similarities))
print(f"Time taken: {elapsed_time:.2f} seconds")

candidate_similarities.head()

Candidate pairs scored: 134606
Time taken: 3.73 seconds


,game_a,game_b,jaccard_similarity
0,42140,93200,0.064140
1,291010,92100,0.243346
2,200170,4500,0.020059
3,330830,65980,0.043324
4,306040,350810,0.113208


In [34]:
OUTPUT_PATH = Path("../data/processed/game_similarity_candidates.parquet")

candidate_similarities.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved candidate similarities to:")
print(OUTPUT_PATH)
print("Shape:", candidate_similarities.shape)

Saved candidate similarities to:
..\data\processed\game_similarity_candidates.parquet
Shape: (134606, 3)


## Inspect the strongest game similarities

As a final sanity check, we inspect the candidate pairs with the highest exact Jaccard similarity.

If the pipeline behaves sensibly, the strongest pairs should represent games with very similar player audiences.

In [35]:
top_similarities = (
    candidate_similarities
    .sort_values("jaccard_similarity", ascending=False)
    .head(20)
)

top_similarities

,game_a,game_b,jaccard_similarity
106114,457820,467000,0.893939
37354,31180,31190,0.833333
124283,8290,8300,0.818182
97782,448640,467000,0.814286
30628,31200,31210,0.804878
128382,31230,31240,0.789062
66581,31250,31260,0.772277
87002,94510,94520,0.770115
123995,31190,31200,0.768421
49884,94520,94530,0.754491


## Add game names to the strongest similarities

To make the strongest behavioural similarities easier to interpret, we attach the game titles from the cleaned metadata table.

In [37]:
# Use game names from the interaction dataset because every game in our
# behavioural analysis is represented here, including games without metadata.
game_names = (
    interactions[["game_id", "game_name"]]
    .drop_duplicates(subset=["game_id"])
)


top_similarities_named = (
    top_similarities
    .merge(
        game_names.rename(
            columns={
                "game_id": "game_a",
                "game_name": "game_a_name"
            }
        ),
        on="game_a",
        how="left"
    )
    .merge(
        game_names.rename(
            columns={
                "game_id": "game_b",
                "game_name": "game_b_name"
            }
        ),
        on="game_b",
        how="left"
    )
)

top_similarities_named[
    [
        "game_a",
        "game_a_name",
        "game_b",
        "game_b_name",
        "jaccard_similarity"
    ]
]

,game_a,game_a_name,game_b,game_b_name,jaccard_similarity
0,457820,Outrage,467000,Insincere,0.893939
1,31180,Tales of Monkey Island: Chapter 2 - The Siege ...,31190,Tales of Monkey Island: Chapter 3 - Lair of th...,0.833333
2,8290,Sam & Max 204: Chariots of the Dogs,8300,Sam & Max 205: What's New Beelzebub?,0.818182
3,448640,Elements II: Hearts of Light,467000,Insincere,0.814286
4,31200,Tales of Monkey Island: Chapter 4 - The Trial ...,31210,Tales of Monkey Island: Chapter 5 - Rise of th...,0.804878
5,31230,Sam & Max 302: The Tomb of Sammun-Mak,31240,Sam & Max 303: They Stole Max's Brain!,0.789062
6,31250,Sam & Max 304: Beyond the Alley of the Dolls,31260,Sam & Max 305: The City that Dares not Sleep,0.772277
7,94510,Back to the Future: Ep 3 - Citizen Brown,94520,Back to the Future: Ep 4 - Double Visions,0.770115
8,31190,Tales of Monkey Island: Chapter 3 - Lair of th...,31200,Tales of Monkey Island: Chapter 4 - The Trial ...,0.768421
9,94520,Back to the Future: Ep 4 - Double Visions,94530,Back to the Future: Ep 5 - OUTATIME,0.754491


### Strongest similarity pairs

The strongest behavioural similarities are dominated by games from the same episodic series, including *Tales of Monkey Island*, *Sam & Max*, and *Back to the Future*.

This is a useful sanity check because the similarity model does not use titles, genres, tags, or other metadata. These relationships emerge purely from overlap between engaged player audiences.

Some highly similar games are not obviously related from their titles alone, which suggests that the behavioural representation may also capture less obvious relationships between player communities.